# Build `gold.nyc_tlc.monthly_stats` with PySpark

This notebook reads the trusted Silver table `silver.nyc_tlc.trips` and publishes a simple Gold aggregate table `gold.nyc_tlc.monthly_stats`.

## Notebook role in the lakehouse

- Bronze keeps raw landed files
- Silver publishes the trusted trip-level analytical contract
- Gold publishes business-ready aggregates for reporting and consumer use

## Business purpose

Publish simple monthly statistics from the trusted Silver trip table for reporting, demos, and downstream business analysis.

## Target grain

**One row per pickup month and source dataset.**

## Intended consumers

- Business analysts
- BI and dashboard users
- Showcase consumers who need monthly trend reporting instead of trip-level detail

## Why this table belongs in Gold

This table is an aggregated, consumer-facing, business-ready dataset built only from Silver.

## Current environment note

In the current environment, SeaweedFS may fail during Iceberg multipart upload initialization.

The typical error is `CreateMultipartUpload` with `403 Access Denied`.

To reduce the risk of that failure, this notebook keeps the Gold write logic intentionally simple and writes a very small unpartitioned aggregate table.


## Inputs, outputs, and dependencies

### Input

- `silver.nyc_tlc.trips`

### Output

- `gold.nyc_tlc.monthly_stats`

### Assumptions

- Spark, Iceberg, Polaris REST catalog, and object-storage access are already configured outside the notebook
- Sensitive Spark options are intentionally hidden in `spark-defaults`
- `silver.nyc_tlc.trips` is the trusted conformed source for Gold

### Important modeling note

This notebook derives `pickup_month` from `pickup_ts` and aggregates by actual pickup month.

This is a business-oriented Gold modeling choice.


## Metrics published in this Gold table

- `trip_count`
- `passenger_count_sum`
- `trip_distance_sum`
- `trip_distance_avg`
- `trip_duration_minutes_avg`
- `fare_amount_sum`
- `tip_amount_sum`
- `total_amount_sum`
- `total_amount_avg`

## Metric interpretation note

`total_amount_sum` is the sum of the `total_amount` field published in Silver.

Use it as a dataset-based monthly collected amount metric, not as a universal revenue truth across all payment behaviors.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [ ]:
SILVER_CATALOG = "silver"
GOLD_CATALOG = "gold"
NAMESPACE = "nyc_tlc"
SOURCE_TABLE = f"{SILVER_CATALOG}.{NAMESPACE}.trips"
FULL_TABLE_NAME = f"{GOLD_CATALOG}.{NAMESPACE}.monthly_stats"

print("SOURCE_TABLE:", SOURCE_TABLE)
print("FULL_TABLE_NAME:", FULL_TABLE_NAME)

## Start Spark

This notebook only sets notebook-local values that are safe to make visible.

Catalog wiring, Iceberg extensions, REST catalog endpoints, and object-storage access are assumed to be injected outside the notebook.


In [ ]:
polaris_credential = "my-polaris-spark-etl-app:mySparkAppSecret"

spark = (
    SparkSession.builder
    .appName("NYC Tripdata - Gold - Monthly Stats - PySpark")
    .config(f"spark.sql.catalog.{SILVER_CATALOG}.credential", polaris_credential)
    .config(f"spark.sql.catalog.{GOLD_CATALOG}.credential", polaris_credential)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "640m")
    .config("spark.executor.cores", 1)
    .config("spark.kubernetes.executor.request.cores", 1)
    .config("spark.kubernetes.executor.limit.cores", 1)
    .getOrCreate()
)

In [ ]:
print("Spark version:", spark.version)
spark.sql("USE silver")
spark.sql("USE gold")
spark.sql("SHOW CATALOGS").show(truncate=False)

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME} PURGE")

## Read the Silver source table

Gold must read only from Silver.

This keeps the Gold table aligned with medallion responsibilities and avoids bypassing the trusted conformance layer.


In [ ]:
silver_trips = spark.table(SOURCE_TABLE)

silver_trips.printSchema()
silver_trips.show(5, truncate=False)

## Build monthly statistics

This Gold table is intentionally simple.

It derives `pickup_month` from `pickup_ts`, groups by `pickup_month` and `source_dataset`, and computes a compact set of monthly metrics.

To keep the write profile small in the current SeaweedFS environment, the staged result is coalesced to one partition before publication.

That is an environment-specific operational choice, not a general Iceberg best practice for all workloads.


In [ ]:
monthly_stats_stage = (
    silver_trips
    .where(F.col("pickup_ts").isNotNull() & F.col("source_dataset").isNotNull())
    .withColumn("pickup_month", F.date_format("pickup_ts", "yyyy-MM"))
    .groupBy("pickup_month", "source_dataset")
    .agg(
        F.count(F.lit(1)).alias("trip_count"),
        F.sum(F.coalesce(F.col("passenger_count"), F.lit(0))).cast("long").alias("passenger_count_sum"),
        F.sum(F.coalesce(F.col("trip_distance"), F.lit(0.0))).alias("trip_distance_sum"),
        F.avg("trip_distance").alias("trip_distance_avg"),
        F.avg("trip_duration_minutes").alias("trip_duration_minutes_avg"),
        F.sum(F.coalesce(F.col("fare_amount"), F.lit(0.0))).alias("fare_amount_sum"),
        F.sum(F.coalesce(F.col("tip_amount"), F.lit(0.0))).alias("tip_amount_sum"),
        F.sum(F.coalesce(F.col("total_amount"), F.lit(0.0))).alias("total_amount_sum"),
        F.avg("total_amount").alias("total_amount_avg")
    )
    .withColumn("load_ts", F.current_timestamp())
    .orderBy("pickup_month", "source_dataset")
    .coalesce(1)
)

monthly_stats_stage.printSchema()
monthly_stats_stage.show(50, truncate=False)

## Pre-write validation

These checks keep validation lightweight and focused on the Gold grain.

- one row should exist per `pickup_month` and `source_dataset`
- monthly trip counts should reconcile to grouped Silver counts


In [ ]:
monthly_stats_row_count = monthly_stats_stage.count()
print("Gold staged rows:", monthly_stats_row_count)

monthly_stats_stage.groupBy("pickup_month", "source_dataset").count().where(F.col("count") > 1).show(truncate=False)


## Publish the Gold Iceberg table

### Write strategy

This notebook performs a full refresh using `CREATE OR REPLACE TABLE`.

### Partition strategy

This Gold table is intentionally left unpartitioned.

Because the dataset is very small after monthly aggregation, an unpartitioned table keeps the write path simple and reduces file-management overhead.

In the current SeaweedFS environment, that also helps keep the write profile conservative.


In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {GOLD_CATALOG}.{NAMESPACE}")

monthly_stats_stage.createOrReplaceTempView("monthly_stats_stage")

spark.sql(f'''
CREATE OR REPLACE TABLE {FULL_TABLE_NAME}
USING iceberg
TBLPROPERTIES (
  'format-version' = '2',
  'comment' = 'Monthly business-ready statistics built from silver.nyc_tlc.trips'
)
AS
SELECT
  pickup_month,
  source_dataset,
  trip_count,
  passenger_count_sum,
  trip_distance_sum,
  trip_distance_avg,
  trip_duration_minutes_avg,
  fare_amount_sum,
  tip_amount_sum,
  total_amount_sum,
  total_amount_avg,
  load_ts
FROM monthly_stats_stage
''')

print(f"Published table: {FULL_TABLE_NAME}")

## Post-write checks

These checks confirm that the Gold table exists and matches the staged monthly aggregate.


In [ ]:
published_count = spark.table(FULL_TABLE_NAME).count()

print("Expected published rows:", monthly_stats_row_count)
print("Actual published rows:", published_count)

if published_count != monthly_stats_row_count:
    raise AssertionError(
        f"Published row count mismatch: expected {monthly_stats_row_count}, got {published_count}"
    )

spark.sql(f"SHOW TABLES IN {GOLD_CATALOG}.{NAMESPACE}").show(truncate=False)
spark.sql(f"DESCRIBE TABLE {FULL_TABLE_NAME}").show(200, truncate=False)

spark.sql(f'''
SELECT
  pickup_month,
  source_dataset,
  trip_count,
  passenger_count_sum,
  trip_distance_sum,
  fare_amount_sum,
  tip_amount_sum,
  total_amount_sum
FROM {FULL_TABLE_NAME}
ORDER BY pickup_month, source_dataset
''').show(200, truncate=False)

try:
    spark.sql(f"SELECT * FROM {FULL_TABLE_NAME}.snapshots").show(truncate=False)
except Exception as exc:
    print("Snapshot metadata query skipped:", exc)

In [ ]:
spark.stop()

## What comes next

This Gold table gives a simple monthly reporting layer.

Other Gold notebooks can build consumer-focused datasets such as zone demand and revenue reporting from the same trusted Silver source.
